# RM-294 Optimization II - Airline Ticket Price Dynamic Programming

Suppose an airline wants to choose what price to set its tickets, given two ticket classes (coach, first class). Coach has three price options (low, medium, and high), while first class has two prices (low and high). 


Customers who buy a ticket may not show up to their own flight. Then, the airline may choose to overbook for coach seats, subject to overbooking costs, this could be bumping people up from coach to first class or paying them a fee.

Each day, only up to one ticket is sold in each of coach and first class. If first class if fully booked, the chance of sale in coach increase by 4%. 

Suppose The following:  

– You have 365 days to sell tickets  
– There are 100 seasts in coach, initially we can oversell 5 coach seats  
– 20 seats in first class  
– $P$(coach customer shows up) = 0.95  
– $P$(first class customer shows up) = 0.97  
– Price for coach tickets is low, medium, or high. That is $p_{coach} \in [l_c, m_c, h_c]$, $p_{coach} \in [300, 325, 350]$  
– Price for first class tickets is low or high. That is $p_{first\ class} \in [l_{f}, h_{f}]$, $p_{first\ class} \in [425, 500]$


– The expected demand for each coach price is: $\mathbb{E}(d_{l_c}) = 0.65$, $\mathbb{E}(d_{m_c}) = 0.45$, and $\mathbb{E}(d_{h_c}) = 0.30$  
– The expected demand for each first class price is: $\mathbb{E}(d_{l_{f}}) = 0.08$ and $\mathbb{E}(d_{h_{f}}) = 0.04$  
– The cost of overbooking, $c$, is $50 to upgrade a coach ticket to first class and $425 to bump a coach ticket off the plane.  
– Discount rate, $r$, is 17% per year

---

### 1. Expected Discounted Profit

In [27]:
import numpy as np
import scipy.stats

In [28]:
# a. Terminal condition

def calculate_terminal_costs(tickets_sold, max_tickets, show_up_rate):
    """
    Calculate expected cost on the day the flight takes off (t = T) using the binomial distribution.

    Parameters:
    - tickets_sold: Dictionary containing the number of tickets sold for each class (e.g., {"coach": 110, "first": 20}).
    - max_tickets: Dictionary containing the maximum number of seats available for each class (e.g., {"coach": 100, "first": 25}).
    - show_up_rate: Dictionary containing the show-up rate for each class (e.g., {"coach": 0.9, "first": 0.95}).

    Returns:
    - float: The expected cost of overbooking on the day the flight takes off.
    """

    # Create a list of each possible situation (no one shows up to everyone shows up) for each ticket class
    k_c = np.arange(0, tickets_sold["coach"] + 1)
    k_f = np.arange(0, tickets_sold["first"] + 1)  

    # Get the probability mass functions for coach and first class
    pmf_c = scipy.stats.binom.pmf(k_c, tickets_sold["coach"], show_up_rate["coach"])
    pmf_f = scipy.stats.binom.pmf(k_f, tickets_sold["first"], show_up_rate["first"])

    expected_cost = 0

    # Sum over every possibilty of attendance:
    for i in k_c:
        for j in k_f:
            joint_probability = pmf_c[i] * pmf_f[j]

            # Find the number of overbooked tickets we upgrade or kick off the plane
            open_firstclass_seats = max_tickets["first"] - j
            overbooked_seats = max(i - max_tickets["coach"], 0)
            upgraded_seats = min(overbooked_seats, open_firstclass_seats)
            bumped_off_plane = overbooked_seats - upgraded_seats     
            
            # Add the expected cost of this scenario to the running total
            expected_cost += joint_probability * (upgraded_seats * 50 + 425 * bumped_off_plane)

    return expected_cost

In [ ]:
# b. Bellman Eqn. / Value function

def expected_profit(t, tickets_sold, ticket_prices, ticket_demand, delta, show_up_rate, max_tickets, overbook):
    """
    Compute the maximum expected discounted profit from time t onward, given the
    number of coach and first-class tickets already sold.

    Uses the Bellman equation recursively, choosing the optimal coach and first-class prices
    to maximize expected revenue minus discounted future overbooking costs.

    Parameters:
    - t: int, current day (0 to 365). Day 365 is departure.
    - tickets_sold: Dictionary containing the number of tickets sold for each class (e.g., {"coach": 110, "first": 20}).
    - ticket_prices: Dictionary containing the lists of price options for each ticket (e.g., {"coach": [300, 325], "first": [425, 500]}).
    - ticket_demand: Dictionary containing the lists of demands for each ticket option (e.g., {"coach": [0.65, 0.45], "first": [0.08, 0.04]}).
    - delta: float, daily discount factor (e.g., 1/(1 + 0.17/365)).
    - show_up_rate: Dictionary containing the show-up probability per class (e.g., {"coach": 0.95, "first": 0.97}).
    - max_tickets: Dictionary containing the maximum number of seats available for each class (e.g., {"coach": 100, "first": 25}).
    - overbook: The number of coach seats that can be overbooked.

    Returns:
    - float: maximum expected discounted profit from this day forward.
    """

    # Since there are only costs on the final day, we can precompute all possible terminal costs
    # as a (max_coach + overbook + 1) x (max_first + 1) matrix
    cost_matrix = np.zeros((max_tickets['coach'] + overbook + 1, max_tickets['first'] + 1))
    for i in range(cost_matrix.shape[0]):
        for j in range(cost_matrix.shape[1]):
            cost_matrix[i, j] = calculate_terminal_costs(tickets_sold={'coach':i,'first':j}, max_tickets=max_tickets, show_up_rate=show_up_rate)

    # If we save each value in a tensor as we calculate it, we can avoid recomputing the same thing repeatedly
    value_tensor = np.full((366, max_tickets['coach'] + overbook + 1, max_tickets['first'] + 1), np.nan)
    value_tensor[365, :, :] = - cost_matrix

    # Now we loop over each day backwards
    for day in reversed(range(value_tensor.shape[0])):
        for c in range(value_tensor.shape[1]):
            for f in range(value_tensor.shape[2]):

                # for each price combination we find the expected value, then choose the best of those
                # as the value for the value_tensor
                best_so_far = -np.inf

                for ci in range(len(ticket_prices['coach'])):
                    cp = ticket_prices['coach'][ci]
                    c_prob = ticket_demand['coach'][ci]

                    for fi in range(len(ticket_prices['first'])):
                        fp = ticket_prices['first'][fi]
                        f_prob = ticket_demand['first'][fi]

                        coach_tickets_available = c < max_tickets['coach'] + overbook
                        first_tickets_available = f < max_tickets['first']

                        # TODO: Save the optimal actions simultaneously, also calculate the value fn for each of the four outcomes P_sale / P_ns, save best


    
    return 0

IndentationError: expected an indented block after 'for' statement on line 37 (2178732587.py, line 39)

Revisiting the original question:  

What is the expected discounted profit over the course of the year?

In [26]:
print(f"The expected profit is: {expected_profit(t=0, tickets_sold={"coach":0, "first":0}, ticket_prices={"coach":[300, 325, 350], "first":[425, 500]},
                                ticket_demand={"coach":[0.65, 0.45, 0.3], "first":[0.08, 0.04]}, delta=0.17, show_up_rate={"coach":0.95, "first":0.97}, 
                                max_tickets={'coach':105,'first':20}, overbook=5)}")

The expected profit is: 0
